# Tests de helpers de OP-03: fix_trips_correspondence

Este notebook se usa para probar helpers y bloques internos de `fix_trips_correspondence()`
antes de pasar a pruebas de integración de la función pública completa.

Objetivo:
- verificar minifuncionalidades de forma aislada
- detectar errores de implementación temprano
- dejar una base fácil de portar después a pytest

Convenciones:
- los tests usan `assert`
- cuando una prueba necesita inspección visual, se puede acompañar con `display(...)`
- estas pruebas no reemplazan los tests integrados de OP-03

## Bloque 0. Preparación

### Imports generales

In [1]:
import json
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd

### Imports del modulo

In [2]:
from pylondrina.schema import DomainSpec, FieldSpec, TripSchema
from pylondrina.reports import Issue
from pylondrina.errors import FixError
from pylondrina.fixing import (
    sanitize_correspondence_context,
    _sanitize_context_with_issues,
    apply_field_corrections,
    _resolve_field_corrections,
    apply_value_corrections,
    _resolve_value_corrections,
    _rebuild_domains_effective_for_fields,
)

### Helpers de apoyo para test

In [3]:
def show_ok(label: str) -> None:
    print(f"OK - {label}")


def assert_json_safe(obj, label: str = "object") -> None:
    try:
        json.dumps(obj)
    except Exception as e:
        raise AssertionError(f"{label} no es JSON-safe: {e}") from e


def get_issue_codes(issues):
    return [issue.code for issue in issues]


def assert_issue_present(issues, code: str) -> None:
    codes = get_issue_codes(issues)
    assert code in codes, f"No se encontró el issue {code}. Codes actuales: {codes}"


def assert_issue_absent(issues, code: str) -> None:
    codes = get_issue_codes(issues)
    assert code not in codes, f"Se encontró inesperadamente el issue {code}. Codes actuales: {codes}"


def assert_counts_by_level(issues, *, errors=None, warnings=None, info=None) -> None:
    counts = {"error": 0, "warning": 0, "info": 0}
    for issue in issues:
        counts[issue.level] = counts.get(issue.level, 0) + 1

    if errors is not None:
        assert counts["error"] == errors, f"errors esperado={errors}, actual={counts['error']}"
    if warnings is not None:
        assert counts["warning"] == warnings, f"warnings esperado={warnings}, actual={counts['warning']}"
    if info is not None:
        assert counts["info"] == info, f"info esperado={info}, actual={counts['info']}"

In [4]:
def make_field(
    name: str,
    dtype: str,
    *,
    required: bool = False,
    constraints: dict | None = None,
    domain: DomainSpec | None = None,
) -> FieldSpec:
    return FieldSpec(
        name=name,
        dtype=dtype,
        required=required,
        constraints=constraints,
        domain=domain,
    )


def make_trip_schema(fields: list[FieldSpec], *, version: str = "1.1") -> TripSchema:
    return TripSchema(
        version=version,
        fields={f.name: f for f in fields},
        required=[f.name for f in fields if f.required],
        semantic_rules=None,
    )


BASE_FIX_SCHEMA = make_trip_schema([
    make_field("movement_id", "string", required=True),
    make_field("user_id", "string", required=True),
    make_field(
        "mode",
        "categorical",
        domain=DomainSpec(values=["walk", "bus", "metro", "car", "unknown"], extendable=True),
    ),
    make_field(
        "purpose",
        "categorical",
        domain=DomainSpec(values=["work", "study", "shopping", "unknown"], extendable=True),
    ),
    make_field("trip_weight", "float"),
])

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 120)

show_ok("Bloque 0 cargado")

OK - Bloque 0 cargado


### Configuración visual

In [6]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 120)

print("Imports OK")
show_ok("Sección 0 cargada")

Imports OK
OK - Sección 0 cargada


## Bloque 1. Contexto de correspondencias

Qué se prueba:
- saneamiento puro de `correspondence_context`
- descarte de keys desconocidas
- descarte de fragmentos no serializables
- emisión de warnings degradables en el helper interno con issues

### Test 1.1 - sanitize_correspondence_context happy path

Qué prueba:
Que el helper puro conserve solo un contexto permitido y ya serializable,
sin introducir cambios innecesarios.

In [7]:
context_ok = {
    "reason": "corrección manual tras revisión",
    "author": "sebastian",
    "source": {"type": "catalog", "name": "EOD_modo", "version": "3.1"},
    "scope": {"fields": ["mode", "purpose"], "rows": {"n_target_rows": 12}},
    "notes": "ajuste de categorías a canónicos",
}

context_sanitized, unknown_keys, dropped_paths = sanitize_correspondence_context(context_ok)

assert context_sanitized == context_ok
assert unknown_keys == []
assert dropped_paths == []
assert_json_safe(context_sanitized, "context_sanitized")

show_ok("Test 1.1 - sanitize_correspondence_context happy path")

OK - Test 1.1 - sanitize_correspondence_context happy path


### Test 1.2 - sanitize_correspondence_context con unknown keys y fragmentos no serializables

Qué prueba:
Que el helper puro:
- descarte claves fuera del whitelist top-level
- descarte solo los fragmentos no serializables
- preserve las partes válidas del contexto

In [12]:
context_mixed = {
    "reason": "ajuste manual",
    "foo": 123,  # key no permitida
    "scope": {
        "fields": ["mode"],
        "rows": {"n_target_rows": 10, "bad": object()},  # fragmento no serializable
    },
    "notes": {"valid": "texto", "bad": object()},  # fragmento no serializable
}

context_sanitized, unknown_keys, dropped_paths = sanitize_correspondence_context(context_mixed)

assert unknown_keys == ["foo"]
assert "reason" in context_sanitized
assert "scope" in context_sanitized
assert context_sanitized["scope"]["fields"] == ["mode"]
assert "rows" in context_sanitized["scope"]
assert "n_target_rows" in context_sanitized["scope"]["rows"]
assert "bad" not in context_sanitized["scope"]["rows"]
assert context_sanitized["notes"]["valid"] == "texto"
assert "bad" not in context_sanitized["notes"]

assert any(path.startswith("scope.rows.bad") for path in dropped_paths)
assert any(path.startswith("notes.bad") for path in dropped_paths)

assert_json_safe(context_sanitized, "context_sanitized")

display(context_sanitized)
show_ok("Test 1.2 - sanitize_correspondence_context mixto")

{'reason': 'ajuste manual',
 'scope': {'fields': ['mode'], 'rows': {'n_target_rows': 10}},
 'notes': {'valid': 'texto'}}

OK - Test 1.2 - sanitize_correspondence_context mixto


### Test 1.3 - sanitize_correspondence_context root inválido

Qué prueba:
Que un root distinto de dict/None falle inmediatamente.

In [13]:
try:
    sanitize_correspondence_context(["not", "a", "dict"])
    raise AssertionError("Se esperaba TypeError por root inválido.")
except TypeError:
    pass

show_ok("Test 1.3 - sanitize_correspondence_context root inválido")

OK - Test 1.3 - sanitize_correspondence_context root inválido


### Test 1.4 - _sanitize_context_with_issues emite warnings degradables

Qué prueba:
Que el helper interno:
- devuelva el contexto saneado
- emita warning por keys desconocidas
- emita warning por fragmentos no serializables

In [14]:
context_with_issues = {
    "reason": "ajuste manual",
    "unknown_key": "x",
    "notes": {"ok": "texto", "bad": object()},
}

context_sanitized, issues = _sanitize_context_with_issues(
    context_with_issues,
    sample_rows_per_issue=5,
    n_rows_total=12,
)

assert context_sanitized["reason"] == "ajuste manual"
assert context_sanitized["notes"]["ok"] == "texto"
assert "bad" not in context_sanitized["notes"]

assert_issue_present(issues, "FIX.CONTEXT.UNKNOWN_KEYS_DROPPED")
assert_issue_present(issues, "FIX.CONTEXT.NON_SERIALIZABLE_DROPPED")
assert_counts_by_level(issues, errors=0, warnings=2, info=0)

for issue in issues:
    assert_json_safe(issue.details, f"details de {issue.code}")

show_ok("Test 1.4 - _sanitize_context_with_issues con warnings")

OK - Test 1.4 - _sanitize_context_with_issues con warnings


### Test 1.5 - _sanitize_context_with_issues root inválido debe abortar

Qué prueba:
Que el helper interno traduzca un contexto raíz inválido a la excepción operativa de Fix.

In [15]:
try:
    _sanitize_context_with_issues(
        ["not", "a", "dict"],
        sample_rows_per_issue=5,
        n_rows_total=3,
    )
    raise AssertionError("Se esperaba FixError por context root inválido.")
except FixError as e:
    assert e.code == "FIX.CONTEXT.INVALID_ROOT"

show_ok("Test 1.5 - _sanitize_context_with_issues fatal")

OK - Test 1.5 - _sanitize_context_with_issues fatal


## Bloque 2. Field corrections

Qué se prueba:
- renombrado puro sin side effects
- política de resolución de field_corrections
- actualización de field_correspondence efectivo
- issues por reglas no aplicables o parcialmente aplicables

### Test 2.1 - apply_field_corrections happy path y no mutación

Qué prueba:
Que el helper puro renombre columnas correctamente y no muta el dataframe original en el camino normal.

In [19]:
df_fields = pd.DataFrame({
    "movement_id": ["m1", "m2"],
    "user_id": ["u1", "u2"],
    "modo": ["BUS", "walk"],
    "proposito": ["work", "study"],
})

df_fields_original = df_fields.copy(deep=True)

df_fields_out = apply_field_corrections(
    df_fields,
    {"modo": "mode", "proposito": "purpose"},
)

assert list(df_fields_out.columns) == ["movement_id", "user_id", "mode", "purpose"]
assert list(df_fields_out.index) == [0, 1]
assert df_fields.equals(df_fields_original)

display(df_fields_out)
show_ok("Test 2.1 - apply_field_corrections happy path")

,movement_id,user_id,mode,purpose
0,m1,u1,BUS,work
1,m2,u2,walk,study


OK - Test 2.1 - apply_field_corrections happy path


### Test 2.2 - _resolve_field_corrections happy path

Qué prueba:
Que el resolver:
- detecte reglas aplicables
- actualice field_correspondence a vista efectiva final canónico -> origen
- derive correctamente fields_effective
- no emita issues cuando todo es válido

In [20]:
df_fields = pd.DataFrame({
    "movement_id": ["m1", "m2"],
    "user_id": ["u1", "u2"],
    "modo": ["BUS", "walk"],
    "proposito": ["work", "study"],
})

applicable, field_corr_final, fields_effective, issues, applied_count, semantic_change = _resolve_field_corrections(
    df_fields,
    schema=BASE_FIX_SCHEMA,
    field_corrections={"modo": "mode", "proposito": "purpose"},
    field_correspondence_current={"movement_id": "movement_id", "user_id": "user_id"},
    sample_rows_per_issue=5,
    n_rows_total=len(df_fields),
)

assert applicable == {"modo": "mode", "proposito": "purpose"}
assert field_corr_final["mode"] == "modo"
assert field_corr_final["purpose"] == "proposito"
assert "modo" not in field_corr_final
assert "proposito" not in field_corr_final
assert set(fields_effective) == {"movement_id", "user_id", "mode", "purpose"}
assert issues == []
assert applied_count == 2
assert semantic_change is True

show_ok("Test 2.2 - _resolve_field_corrections happy path")

OK - Test 2.2 - _resolve_field_corrections happy path


### Test 2.3 - _resolve_field_corrections con source faltante y aplicación parcial

Qué prueba:
Que una regla inválida por columna origen ausente:
- se degrade a issue recuperable
- no bloquee las otras reglas válidas
- emita además el resumen agregado de partial apply

In [21]:
df_fields = pd.DataFrame({
    "movement_id": ["m1", "m2"],
    "user_id": ["u1", "u2"],
    "modo": ["BUS", "walk"],
})

applicable, field_corr_final, fields_effective, issues, applied_count, semantic_change = _resolve_field_corrections(
    df_fields,
    schema=BASE_FIX_SCHEMA,
    field_corrections={
        "modo": "mode",
        "missing_col": "purpose",
    },
    field_correspondence_current={"movement_id": "movement_id", "user_id": "user_id"},
    sample_rows_per_issue=5,
    n_rows_total=len(df_fields),
)

assert applicable == {"modo": "mode"}
assert field_corr_final["mode"] == "modo"
assert_issue_present(issues, "FIX.FIELD.SOURCE_COLUMN_MISSING")
assert_issue_present(issues, "FIX.FIELD.PARTIAL_APPLY")
assert applied_count == 1
assert semantic_change is True

show_ok("Test 2.3 - _resolve_field_corrections parcial")

OK - Test 2.3 - _resolve_field_corrections parcial


### Test 2.5 - _resolve_field_corrections rechaza canónico -> canónico

Qué prueba:
Que OP-03 no permita usar field_corrections para renombrar entre nombres ya canónicos.

In [ ]:
df_fields_canonical = pd.DataFrame({
    "movement_id": ["m1"],
    "user_id": ["u1"],
    "mode": ["bus"],
    "purpose": ["work"],
})

applicable, field_corr_final, fields_effective, issues, applied_count, semantic_change = _resolve_field_corrections(
    df_fields_canonical,
    schema=BASE_FIX_SCHEMA,
    field_corrections={"mode": "purpose"},
    field_correspondence_current={},
    sample_rows_per_issue=5,
    n_rows_total=len(df_fields_canonical),
)

assert applicable == {}
assert_issue_present(issues, "FIX.FIELD.RULE_NOT_ALLOWED")
assert applied_count == 0
assert semantic_change is False

show_ok("Test 2.5 - _resolve_field_corrections canónico a canónico")

OK - Test 2.5 - _resolve_field_corrections canónico a canónico


## Bloque 3. Value corrections

Qué se prueba:
- recodificación pura sin side effects
- política de resolución de value_corrections
- actualización de value_correspondence efectivo
- issues por campos incompatibles, faltantes o reglas sin efecto

### Test 3.1 - apply_value_corrections happy path y no mutación

Qué prueba:
Que el helper puro recodifique solo donde corresponde, preserve nulos y no muta el dataframe original.

In [24]:
df_values = pd.DataFrame({
    "mode": ["BUS", "WALK", None, "BUS"],
    "purpose": ["work", "study", "work", "shopping"],
    "trip_weight": [1.0, 2.0, 3.0, 4.0],
})

df_values_original = df_values.copy(deep=True)

df_values_out = apply_value_corrections(
    df_values,
    {
        "mode": {"BUS": "bus", "WALK": "walk"},
        "purpose": {"shopping": "shopping"},
    },
)

assert df_values_out["mode"].tolist() == ["bus", "walk", None, "bus"]
assert df_values_out["purpose"].tolist() == ["work", "study", "work", "shopping"]
assert df_values.equals(df_values_original)

display(df_values_out)
show_ok("Test 3.1 - apply_value_corrections happy path")

,mode,purpose,trip_weight
0,bus,work,1.0
1,walk,study,2.0
2,None,work,3.0
3,bus,shopping,4.0


OK - Test 3.1 - apply_value_corrections happy path


### Test 3.2 - _resolve_value_corrections happy path

Qué prueba:
Que el resolver:
- detecte mappings aplicables
- actualice value_correspondence final por campo
- compute replacements_count
- marque touched_fields correctamente

In [25]:
df_values = pd.DataFrame({
    "mode": ["BUS", "WALK", None, "BUS"],
    "purpose": ["work", "study", "work", "shopping"],
})

applicable, value_corr_final, issues, applied_fields_count, replacements_count, touched_fields, semantic_change = _resolve_value_corrections(
    df_values,
    schema=BASE_FIX_SCHEMA,
    value_corrections={
        "mode": {"BUS": "bus", "WALK": "walk"},
    },
    value_correspondence_current={"mode": {"METRO": "metro"}},
    sample_rows_per_issue=5,
    n_rows_total=len(df_values),
)

assert applicable == {"mode": {"BUS": "bus", "WALK": "walk"}}
assert value_corr_final["mode"]["METRO"] == "metro"
assert value_corr_final["mode"]["BUS"] == "bus"
assert value_corr_final["mode"]["WALK"] == "walk"
assert issues == []
assert applied_fields_count == 1
assert replacements_count == 3
assert touched_fields == ["mode"]
assert semantic_change is True

show_ok("Test 3.2 - _resolve_value_corrections happy path")

OK - Test 3.2 - _resolve_value_corrections happy path


### Test 3.3 - _resolve_value_corrections con campo faltante y aplicación parcial

Qué prueba:
Que un campo inexistente:
- se degrade a issue recuperable
- no bloquee otros campos válidos
- emita además el resumen agregado de partial apply

In [26]:
df_values = pd.DataFrame({
    "mode": ["BUS", "WALK", None, "BUS"],
})

applicable, value_corr_final, issues, applied_fields_count, replacements_count, touched_fields, semantic_change = _resolve_value_corrections(
    df_values,
    schema=BASE_FIX_SCHEMA,
    value_corrections={
        "mode": {"BUS": "bus"},
        "missing_field": {"x": "y"},
    },
    value_correspondence_current={},
    sample_rows_per_issue=5,
    n_rows_total=len(df_values),
)

assert applicable == {"mode": {"BUS": "bus"}}
assert_issue_present(issues, "FIX.VALUE.FIELD_MISSING")
assert_issue_present(issues, "FIX.VALUE.PARTIAL_APPLY")
assert applied_fields_count == 1
assert replacements_count == 2
assert touched_fields == ["mode"]
assert semantic_change is True

show_ok("Test 3.3 - _resolve_value_corrections parcial")

OK - Test 3.3 - _resolve_value_corrections parcial


### Test 3.4 - _resolve_value_corrections rechaza campo no categórico

Qué prueba:
Que OP-03 no use value_corrections sobre campos incompatibles.

In [27]:
df_values = pd.DataFrame({
    "trip_weight": [1.0, 2.0, 3.0],
})

applicable, value_corr_final, issues, applied_fields_count, replacements_count, touched_fields, semantic_change = _resolve_value_corrections(
    df_values,
    schema=BASE_FIX_SCHEMA,
    value_corrections={"trip_weight": {1.0: 10.0}},
    value_correspondence_current={},
    sample_rows_per_issue=5,
    n_rows_total=len(df_values),
)

assert applicable == {}
assert_issue_present(issues, "FIX.VALUE.FIELD_NOT_COMPATIBLE")
assert applied_fields_count == 0
assert replacements_count == 0
assert touched_fields == []
assert semantic_change is False

show_ok("Test 3.4 - _resolve_value_corrections no categórico")

OK - Test 3.4 - _resolve_value_corrections no categórico


### Test 3.5 - _resolve_value_corrections con target ya presente y source_value ausente

Qué prueba:
Que el resolver:
- avise cuando la regla colapsa hacia un target ya presente
- avise cuando algunas source rules no tienen match
- siga aplicando lo que sí corresponde

In [28]:
df_values = pd.DataFrame({
    "mode": ["BUS", "bus", "walk", None],
})

applicable, value_corr_final, issues, applied_fields_count, replacements_count, touched_fields, semantic_change = _resolve_value_corrections(
    df_values,
    schema=BASE_FIX_SCHEMA,
    value_corrections={
        "mode": {
            "BUS": "bus",   # target ya presente
            "TAXI": "taxi", # valor fuente ausente
        }
    },
    value_correspondence_current={},
    sample_rows_per_issue=5,
    n_rows_total=len(df_values),
)

assert applicable == {"mode": {"BUS": "bus"}}
assert_issue_present(issues, "FIX.VALUE.TARGET_ALREADY_PRESENT")
assert_issue_present(issues, "FIX.VALUE.SOURCE_VALUES_NOT_FOUND")
assert applied_fields_count == 1
assert replacements_count == 1
assert touched_fields == ["mode"]
assert semantic_change is True

show_ok("Test 3.5 - _resolve_value_corrections warnings específicos")

OK - Test 3.5 - _resolve_value_corrections warnings específicos


## Bloque 4. Rebuild de domains_effective

Qué se prueba:
- reconstrucción desde el estado final del dataset
- actualización solo de campos tocados
- recálculo de extended y added_values
- preservación selectiva de unknown_value y strict_applied

### Test 4.1 - _rebuild_domains_effective_for_fields reconstruye desde el estado final observado

Qué prueba:
Que los entries nuevos se calculen a partir de los valores realmente presentes después del fix,
y no desde snapshots viejos.

In [29]:
df_after_values = pd.DataFrame({
    "mode": ["bus", "walk", "bike", "unknown", None],
    "purpose": ["work", "health", "unknown", "work", None],
})

metadata_domains_before = {
    "mode": {
        "values": ["BUS", "walk", "unknown"],
        "extended": False,
        "unknown_value": "unknown",
        "strict_applied": False,
    },
    "purpose": {
        "values": ["work", "study", "unknown"],
        "extended": False,
        "unknown_value": "unknown",
        "strict_applied": True,
    },
}

schema_effective_domains_before = {
    "mode": {"values": ["BUS", "walk", "unknown"], "extended": False},
    "purpose": {"values": ["work", "study", "unknown"], "extended": False},
}

metadata_domains_updated, schema_domains_updated, updated_fields = _rebuild_domains_effective_for_fields(
    data_after_values=df_after_values,
    touched_fields=["mode", "purpose"],
    schema=BASE_FIX_SCHEMA,
    metadata_domains_effective=metadata_domains_before,
    schema_effective_domains=schema_effective_domains_before,
)

assert updated_fields == ["mode", "purpose"]

assert metadata_domains_updated["mode"]["values"] == ["bike", "bus", "unknown", "walk"]
assert metadata_domains_updated["mode"]["extended"] is True
assert metadata_domains_updated["mode"]["added_values"] == ["bike"]
assert metadata_domains_updated["mode"]["unknown_value"] == "unknown"
assert metadata_domains_updated["mode"]["strict_applied"] is False

assert metadata_domains_updated["purpose"]["values"] == ["health", "unknown", "work"]
assert metadata_domains_updated["purpose"]["extended"] is True
assert metadata_domains_updated["purpose"]["added_values"] == ["health"]
assert metadata_domains_updated["purpose"]["unknown_value"] == "unknown"
assert metadata_domains_updated["purpose"]["strict_applied"] is True

assert schema_domains_updated["mode"] == metadata_domains_updated["mode"]
assert schema_domains_updated["purpose"] == metadata_domains_updated["purpose"]

assert_json_safe(metadata_domains_updated, "metadata_domains_updated")
assert_json_safe(schema_domains_updated, "schema_domains_updated")

show_ok("Test 4.1 - _rebuild_domains_effective_for_fields happy path")

OK - Test 4.1 - _rebuild_domains_effective_for_fields happy path


### Test 4.2 - _rebuild_domains_effective_for_fields actualiza solo touched_fields y salta faltantes

Qué prueba:
Que el helper no toque dominios no afectados y que ignore campos tocados que no existen en el dataframe final.

In [30]:
df_after_values = pd.DataFrame({
    "mode": ["bus", "walk", None],
})

metadata_domains_before = {
    "mode": {
        "values": ["walk", "bus"],
        "extended": False,
    },
    "user_gender": {
        "values": ["female", "male", "unknown"],
        "extended": False,
    },
}

schema_effective_domains_before = {
    "mode": {
        "values": ["walk", "bus"],
        "extended": False,
    },
    "user_gender": {
        "values": ["female", "male", "unknown"],
        "extended": False,
    },
}

metadata_domains_updated, schema_domains_updated, updated_fields = _rebuild_domains_effective_for_fields(
    data_after_values=df_after_values,
    touched_fields=["mode", "missing_field"],
    schema=BASE_FIX_SCHEMA,
    metadata_domains_effective=metadata_domains_before,
    schema_effective_domains=schema_effective_domains_before,
)

assert updated_fields == ["mode"]
assert metadata_domains_updated["user_gender"] == metadata_domains_before["user_gender"]
assert schema_domains_updated["user_gender"] == schema_effective_domains_before["user_gender"]
assert metadata_domains_updated["mode"]["values"] == ["bus", "walk"]

show_ok("Test 4.2 - _rebuild_domains_effective_for_fields touched_fields selectivo")

OK - Test 4.2 - _rebuild_domains_effective_for_fields touched_fields selectivo


## Bloque final: Smoke tests integrados de fix_trips_correspondence

Esta sección usa la función pública completa `fix_trips_correspondence(...)`
sobre setups simples y controlados.

Objetivo:
- verificar integración básica de helpers + función pública
- comprobar resultado tabular, reporte y trazabilidad mínima
- detectar bugs evidentes antes de pasar a tests de integración más amplios

### Preparación

In [31]:
from copy import deepcopy

import pandas as pd

from pylondrina.schema import FieldSpec, DomainSpec, TripSchema, TripSchemaEffective
from pylondrina.datasets import TripDataset
from pylondrina.reports import OperationReport
from pylondrina.errors import FixError
from pylondrina.fixing import FixCorrespondenceOptions, fix_trips_correspondence


def make_field(
    name: str,
    dtype: str,
    *,
    required: bool = False,
    constraints: dict | None = None,
    domain: DomainSpec | None = None,
) -> FieldSpec:
    return FieldSpec(
        name=name,
        dtype=dtype,
        required=required,
        constraints=constraints,
        domain=domain,
    )


BASE_FIX_SCHEMA_SMOKE = TripSchema(
    version="1.1",
    fields={
        "movement_id": make_field("movement_id", "string", required=True),
        "user_id": make_field("user_id", "string", required=True),
        "mode": make_field(
            "mode",
            "categorical",
            required=False,
            domain=DomainSpec(values=["walk", "bus", "metro", "car", "unknown"], extendable=True),
        ),
        "purpose": make_field(
            "purpose",
            "categorical",
            required=False,
            domain=DomainSpec(values=["work", "study", "shopping", "health", "unknown"], extendable=True),
        ),
        "trip_weight": make_field("trip_weight", "float", required=False),
    },
    required=["movement_id", "user_id"],
    semantic_rules=None,
)


def _build_base_domains_effective(schema: TripSchema, fields_present: list[str]) -> dict:
    out = {}
    for field_name in fields_present:
        fs = schema.fields.get(field_name)
        if fs is None or fs.domain is None:
            continue
        out[field_name] = {
            "values": sorted(list(fs.domain.values)),
            "extended": False,
            "added_values": [],
            "unknown_value": "unknown" if "unknown" in fs.domain.values else None,
            "strict_applied": False,
        }
    return out


def make_tripdataset_for_fix_smoke(
    df: pd.DataFrame,
    *,
    schema: TripSchema | None = None,
    is_validated: bool = True,
    dataset_id: str = "ds-fix-smoke-001",
    field_correspondence: dict | None = None,
    value_correspondence: dict | None = None,
) -> TripDataset:
    schema = schema or BASE_FIX_SCHEMA_SMOKE

    if field_correspondence is None:
        # Solo deja mapeo identidad para columnas canónicas ya presentes
        field_correspondence = {
            c: c for c in df.columns if c in schema.fields
        }

    if value_correspondence is None:
        value_correspondence = {}

    base_domains_effective = _build_base_domains_effective(
        schema=schema,
        fields_present=[c for c in df.columns if c in schema.fields],
    )

    metadata = {
        "dataset_id": dataset_id,
        "events": [],
        "is_validated": is_validated,
        "mappings": {
            "field_correspondence": deepcopy(field_correspondence),
            "value_correspondence": deepcopy(value_correspondence),
        },
        "domains_effective": deepcopy(base_domains_effective),
    }

    schema_effective = TripSchemaEffective(
        dtype_effective={},
        overrides={},
        domains_effective=deepcopy(base_domains_effective),
        temporal={},
        fields_effective=list(df.columns),
    )

    provenance = {
        "source": {
            "name": "synthetic",
            "entity": "trips",
            "version": "smoke-tests-op03-v1",
        },
        "notes": ["dataset sintético mínimo para smoke tests de OP-03"],
    }

    return TripDataset(
        data=df.copy(deep=True),
        schema=schema,
        schema_version=schema.version,
        provenance=provenance,
        field_correspondence=deepcopy(field_correspondence),
        value_correspondence=deepcopy(value_correspondence),
        metadata=metadata,
        schema_effective=schema_effective,
    )

### Smoke 1 - Happy path integrado fields -> values

Qué prueba:
- la función pública integra bien el pipeline principal
- aplica primero field_corrections y luego value_corrections
- actualiza mappings, domains_effective, schema_effective y evento
- no muta el input

In [36]:
df_ok = pd.DataFrame({
    "movement_id": ["m1", "m2"],
    "user_id": ["u1", "u2"],
    "modo_raw": ["BUS", "WALK"],
    "proposito_raw": ["WORK", "STUDY"],
    "trip_weight": [1.0, 2.0],
})

trips = make_tripdataset_for_fix_smoke(df_ok, is_validated=True)
data_before = trips.data.copy(deep=True)
metadata_before = deepcopy(trips.metadata)

fixed, report = fix_trips_correspondence(
    trips,
    field_corrections={
        "modo_raw": "mode",
        "proposito_raw": "purpose",
    },
    value_corrections={
        "mode": {"BUS": "bus", "WALK": "walk"},
        "purpose": {"WORK": "work", "STUDY": "study"},
    },
    options=FixCorrespondenceOptions(),
    correspondence_context={
        "reason": "ajuste smoke test",
        "notes": "happy path mínimo integrado",
    },
)

assert isinstance(fixed, TripDataset)
assert isinstance(report, OperationReport)

# input intacto
pd.testing.assert_frame_equal(trips.data, data_before)
assert trips.metadata == metadata_before

# output tabular
assert list(fixed.data.columns) == ["movement_id", "user_id", "mode", "purpose", "trip_weight"]
assert fixed.data["mode"].tolist() == ["bus", "walk"]
assert fixed.data["purpose"].tolist() == ["work", "study"]

# summary mínimo
assert report.summary["n_rows"] == 2
assert report.summary["n_field_corrections_requested"] == 2
assert report.summary["n_field_corrections_applied"] == 2
assert report.summary["n_value_corrections_fields_requested"] == 2
assert report.summary["n_value_corrections_fields_applied"] == 2
assert report.summary["n_value_replacements_applied"] == 4
assert report.summary["noop"] is False
assert set(report.summary["domains_effective_updated_fields"]) == {"mode", "purpose"}

# estado lógico
assert fixed.metadata["is_validated"] is False
assert fixed.metadata["dataset_id"] == trips.metadata["dataset_id"]

# correspondencias efectivas finales
assert fixed.field_correspondence["mode"] == "modo_raw"
assert fixed.field_correspondence["purpose"] == "proposito_raw"
assert fixed.value_correspondence["mode"]["BUS"] == "bus"
assert fixed.value_correspondence["purpose"]["WORK"] == "work"

# metadata["mappings"] coherente
assert fixed.metadata["mappings"]["field_correspondence"] == fixed.field_correspondence
assert fixed.metadata["mappings"]["value_correspondence"] == fixed.value_correspondence

# domains_effective actualizado solo donde corresponde
assert fixed.metadata["domains_effective"]["mode"]["values"] == ["bus", "walk"]
assert fixed.metadata["domains_effective"]["purpose"]["values"] == ["study", "work"]
assert fixed.schema_effective.domains_effective["mode"]["values"] == ["bus", "walk"]
assert set(fixed.schema_effective.fields_effective) == set(["movement_id", "user_id", "mode", "purpose", "trip_weight"])

# evento mínimo
assert len(fixed.metadata["events"]) == 1
event = fixed.metadata["events"][-1]
assert event["op"] == "fix_trips_correspondence"
assert event["summary"] == report.summary
assert event["parameters"] == report.parameters
assert "issues_summary" in event
assert "context" in event
assert event["context"]["reason"] == "ajuste smoke test"

display(fixed.data)
display(report.issues)
show_ok("Smoke 1 - happy path integrado")

,movement_id,user_id,mode,purpose,trip_weight
0,m1,u1,bus,work,1.0
1,m2,u2,walk,study,2.0


[Issue(level='info', code='FIX.DOMAINS.UPDATED', message='Se actualizaron los domains_effective para los campos tocados por value_corrections.', field=None, source_field=None, row_count=None, details={'kind': 'domains_effective', 'updated_fields': ['mode', 'purpose'], 'added_values_by_field': {}, 'n_rows_total': 2}),
 Issue(level='info', code='FIX.INFO.FIELD_CORRECTIONS_APPLIED', message='Se aplicaron correcciones de campos (requested=2, applied=2).', field=None, source_field=None, row_count=None, details={'kind': 'field_corrections', 'requested_count': 2, 'applied_count': 2, 'mapping_sample': {'modo_raw': 'mode', 'proposito_raw': 'purpose'}, 'n_rows_total': 2}),
 Issue(level='info', code='FIX.INFO.VALUE_CORRECTIONS_APPLIED', message='Se aplicaron correcciones de valores (requested_fields=2, applied_fields=2, replacements=4).', field=None, source_field=None, row_count=None, details={'kind': 'value_corrections', 'requested_fields_count': 2, 'applied_fields_count': 2, 'replacements_count

OK - Smoke 1 - happy path integrado


### Smoke 2 - Sin cambios efectivos por falta de correcciones

Qué prueba:
- la función pública resuelve el NOOP explícito
- preserva `is_validated`
- igual deja reporte y evento mínimos

In [38]:
df_noop = pd.DataFrame({
    "movement_id": ["m1", "m2"],
    "user_id": ["u1", "u2"],
    "mode": ["bus", "walk"],
    "purpose": ["work", "study"],
})

trips = make_tripdataset_for_fix_smoke(df_noop, is_validated=True)
data_before = trips.data.copy(deep=True)
metadata_before = deepcopy(trips.metadata)

fixed, report = fix_trips_correspondence(
    trips,
    field_corrections=None,
    value_corrections=None,
    options=FixCorrespondenceOptions(),
)

assert isinstance(fixed, TripDataset)
assert isinstance(report, OperationReport)

# input intacto
pd.testing.assert_frame_equal(trips.data, data_before)
assert trips.metadata == metadata_before

# output equivalente
pd.testing.assert_frame_equal(fixed.data, df_noop)

# NOOP real
codes = [i.code for i in report.issues]
assert "FIX.NO_EFFECTIVE_CHANGES.NO_CORRECTIONS" in codes
assert report.summary["noop"] is True
assert fixed.metadata["is_validated"] is True

# evento y parámetros
assert len(fixed.metadata["events"]) == 1
event = fixed.metadata["events"][-1]
assert event["op"] == "fix_trips_correspondence"
assert event["summary"] == report.summary
assert event["parameters"] == report.parameters

display(report)
show_ok("Smoke 2 - sin cambios efectivos")

OperationReport(ok=True, issues=[Issue(level='info', code='FIX.NO_EFFECTIVE_CHANGES.NO_CORRECTIONS', message='No se proporcionaron correcciones; la operación terminó sin cambios efectivos.', field=None, source_field=None, row_count=None, details={'kind': 'noop', 'field_corrections_provided': False, 'value_corrections_provided': False, 'n_rows_total': 2})], summary={'n_rows': 2, 'n_field_corrections_requested': 0, 'n_field_corrections_applied': 0, 'n_value_corrections_fields_requested': 0, 'n_value_corrections_fields_applied': 0, 'n_value_replacements_applied': 0, 'domains_effective_updated_fields': [], 'noop': True, 'limits': {'max_issues': 200, 'issues_truncated': False, 'n_issues_emitted': 1, 'n_issues_detected_total': 1}}, parameters={'field_corrections': None, 'value_corrections': None, 'strict': False, 'max_issues': 200, 'sample_rows_per_issue': 50})

OK - Smoke 2 - sin cambios efectivos


### Smoke 3 - Degradado con warnings y cambio efectivo parcial

Qué prueba:
- la función pública sigue cuando hay reglas recuperables no aplicables
- emite warnings por partial apply y por saneamiento de context
- igual produce salida coherente y cambio efectivo real

In [40]:
df_warn = pd.DataFrame({
    "movement_id": ["m1", "m2"],
    "user_id": ["u1", "u2"],
    "purpose_raw": ["WORK", "WORK"],
    "trip_weight": [1.0, 2.0],
})

trips = make_tripdataset_for_fix_smoke(df_warn, is_validated=True)
data_before = trips.data.copy(deep=True)

fixed, report = fix_trips_correspondence(
    trips,
    field_corrections={
        "purpose_raw": "purpose",
        "missing_mode": "mode",   # source faltante -> warning recuperable
    },
    value_corrections={
        "purpose": {"WORK": "work"},
    },
    options=FixCorrespondenceOptions(strict=False),
    correspondence_context={
        "reason": "ajuste parcial smoke",
        "unknown_key": "debe descartarse",
        "notes": {"ok": "texto", "bad": object()},
    },
)

# input intacto
pd.testing.assert_frame_equal(trips.data, data_before)

# salida con cambio real parcial
assert "purpose" in fixed.data.columns
assert fixed.data["purpose"].tolist() == ["work", "work"]
assert fixed.metadata["is_validated"] is False

codes = [i.code for i in report.issues]
assert "FIX.FIELD.SOURCE_COLUMN_MISSING" in codes
assert "FIX.FIELD.PARTIAL_APPLY" in codes
assert "FIX.CONTEXT.UNKNOWN_KEYS_DROPPED" in codes
assert "FIX.CONTEXT.NON_SERIALIZABLE_DROPPED" in codes

assert report.summary["n_field_corrections_requested"] == 2
assert report.summary["n_field_corrections_applied"] == 1
assert report.summary["n_value_corrections_fields_requested"] == 1
assert report.summary["n_value_corrections_fields_applied"] == 1
assert report.summary["n_value_replacements_applied"] == 2
assert report.summary["noop"] is False

event = fixed.metadata["events"][-1]
assert event["op"] == "fix_trips_correspondence"
assert event["summary"] == report.summary
assert event["parameters"] == report.parameters
assert "unknown_key" not in event["context"]
assert event["context"]["notes"]["ok"] == "texto"
assert "bad" not in event["context"]["notes"]

display(trips.data)
display(fixed.data)
display(report.issues)
show_ok("Smoke 3 - degradado con warnings")

,movement_id,user_id,purpose_raw,trip_weight
0,m1,u1,WORK,1.0
1,m2,u2,WORK,2.0


,movement_id,user_id,purpose,trip_weight
0,m1,u1,work,1.0
1,m2,u2,work,2.0


[Issue(level='warning', code='FIX.CONTEXT.UNKNOWN_KEYS_DROPPED', message='correspondence_context contiene claves no reconocidas; esas claves se descartarán del evento.', field=None, source_field=None, row_count=None, details={'kind': 'context', 'n_rows_total': 2, 'sample_rows_per_issue': 50, 'unknown_keys': ['unknown_key'], 'allowed_keys': ['reason', 'author', 'source', 'scope', 'notes'], 'action': 'drop_unknown_keys'}),
 Issue(level='warning', code='FIX.CONTEXT.NON_SERIALIZABLE_DROPPED', message='correspondence_context contiene valores no serializables; esos fragmentos se descartarán del evento.', field=None, source_field=None, row_count=None, details={'kind': 'context', 'n_rows_total': 2, 'sample_rows_per_issue': 50, 'dropped_paths': ['notes.bad'], 'reason': 'non_serializable_values', 'action': 'drop_non_serializable_fragments'}),
 Issue(level='warning', code='FIX.FIELD.SOURCE_COLUMN_MISSING', message="No se puede aplicar la corrección de campo 'missing_mode' -> 'mode' porque la colu

OK - Smoke 3 - degradado con warnings


### Smoke 4 - Fatal de precondición sin evento ni mutación

Qué prueba:
- una precondición inválida aborta antes del pipeline
- no se muta el input
- no se agregan eventos al dataset original

In [41]:
df_fatal = pd.DataFrame({
    "movement_id": ["m1"],
    "user_id": ["u1"],
    "mode": ["bus"],
})

trips = make_tripdataset_for_fix_smoke(df_fatal, is_validated=True)
data_before = trips.data.copy(deep=True)
metadata_before = deepcopy(trips.metadata)

raised = None
try:
    fix_trips_correspondence(
        trips,
        field_corrections={"mode": "purpose"},  # canónico -> canónico / caso no permitido
        value_corrections=None,
        options=FixCorrespondenceOptions(),
        correspondence_context=["not", "a", "dict"],  # root inválido -> fatal
    )
except Exception as e:
    raised = e

assert raised is not None
assert isinstance(raised, FixError)

# como es fatal de precondición, el input debe quedar intacto
pd.testing.assert_frame_equal(trips.data, data_before)
assert trips.metadata == metadata_before
assert trips.metadata["events"] == []

display(raised)
show_ok("Smoke 4 - fatal de precondición sin side effects")

FixError(message='correspondence_context debe ser dict o None; se recibió un tipo no interpretable.', code='FIX.CONTEXT.INVALID_ROOT', details={'received_type': 'list', 'expected_type': 'dict | None', 'action': 'abort'}, issue=Issue(level='error', code='FIX.CONTEXT.INVALID_ROOT', message='correspondence_context debe ser dict o None; se recibió un tipo no interpretable.', field=None, source_field=None, row_count=None, details={'received_type': 'list', 'expected_type': 'dict | None', 'action': 'abort'}), issues=(Issue(level='error', code='FIX.CONTEXT.INVALID_ROOT', message='correspondence_context debe ser dict o None; se recibió un tipo no interpretable.', field=None, source_field=None, row_count=None, details={'received_type': 'list', 'expected_type': 'dict | None', 'action': 'abort'}),))

OK - Smoke 4 - fatal de precondición sin side effects


### Smoke 5 - Truncamiento básico y consistencia de summary/evento

Qué prueba:
- el límite `max_issues` se refleja en el reporte
- aparece `FIX.CORE.ISSUES_TRUNCATED`
- summary y evento quedan alineados

In [42]:
df_trunc = pd.DataFrame({
    "movement_id": ["m1", "m2"],
    "user_id": ["u1", "u2"],
    "mode": ["bus", "walk"],
    "purpose": ["work", "study"],
})

trips = make_tripdataset_for_fix_smoke(df_trunc, is_validated=True)

fixed, report = fix_trips_correspondence(
    trips,
    field_corrections={
        "missing_1": "mode",
        "missing_2": "purpose",
        "missing_3": "trip_weight",
    },
    value_corrections=None,
    options=FixCorrespondenceOptions(
        strict=False,
        max_issues=2,
        sample_rows_per_issue=5,
    ),
)

codes = [i.code for i in report.issues]
assert "FIX.CORE.ISSUES_TRUNCATED" in codes
assert report.summary["limits"]["issues_truncated"] is True
assert report.summary["limits"]["n_issues_emitted"] <= 2

event = fixed.metadata["events"][-1]
assert event["summary"] == report.summary
assert "issues_summary" in event

display(report)
show_ok("Smoke 5 - truncamiento básico")

OperationReport(ok=True, issues=[Issue(level='warning', code='FIX.FIELD.SOURCE_COLUMN_MISSING', message="No se puede aplicar la corrección de campo 'missing_1' -> 'mode' porque la columna origen no existe en trips.data.", field='mode', source_field='missing_1', row_count=None, details={'kind': 'field_corrections', 'n_rows_total': 2, 'sample_rows_per_issue': 5, 'source_column': 'missing_1', 'target_column': 'mode', 'available_columns_sample': ['movement_id', 'user_id', 'mode', 'purpose'], 'available_columns_total': 4, 'action': 'skip_rule'}), Issue(level='warning', code='FIX.CORE.ISSUES_TRUNCATED', message='Se alcanzó el límite máximo de issues permitido; el reporte fue truncado.', field=None, source_field=None, row_count=None, details={'max_issues': 2, 'n_issues_emitted': 2, 'n_issues_detected_total': 4, 'issues_truncated': True})], summary={'n_rows': 2, 'n_field_corrections_requested': 3, 'n_field_corrections_applied': 0, 'n_value_corrections_fields_requested': 0, 'n_value_corrections

OK - Smoke 5 - truncamiento básico
